# Grok-robotics-06-classical

**Stage 06 — Robotics Classical Control（不用 RL 先能动）**

## 概念
机器人学在 RL 之前已有完整工具链：
1. **正运动学 FK** — 关节角 → 末端位姿
2. **逆运动学 IK** — 目标位姿 → 关节角
3. **PD/PID 控制** — 跟踪参考轨迹
4. **重力补偿**

直观理解：**模型 + 反馈** 即可完成到达任务；RL 是在模型困难/目标稀疏时再上场。


In [ ]:

import json, time
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT=Path("/kaggle/working"); OUT.mkdir(exist_ok=True)
SEED=0; rng=np.random.default_rng(SEED)

gpu_info={"cuda":False,"device_count":0,"names":[]}
try:
    import torch
    gpu_info={"cuda":torch.cuda.is_available(),"device_count":torch.cuda.device_count(),"names":[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else []}
except Exception as e:
    gpu_info["error"]=str(e)
print(gpu_info)

class TwoLinkArm:
    """Planar 2R arm. lengths l1,l2; state [q1,q2,dq1,dq2]."""
    def __init__(self, l1=1.0, l2=1.0, dt=0.01, damping=0.1):
        self.l1,self.l2,self.dt,self.damping=l1,l2,dt,damping
        self.reset()
    def reset(self, q=None):
        self.q = np.array([0.2, -0.4], dtype=np.float64) if q is None else np.array(q,dtype=np.float64)
        self.dq = np.zeros(2)
        return self.q.copy()
    def fk(self, q=None):
        q = self.q if q is None else q
        x = self.l1*np.cos(q[0]) + self.l2*np.cos(q[0]+q[1])
        y = self.l1*np.sin(q[0]) + self.l2*np.sin(q[0]+q[1])
        return np.array([x,y])
    def jac(self, q=None):
        q = self.q if q is None else q
        j11=-self.l1*np.sin(q[0])-self.l2*np.sin(q[0]+q[1])
        j12=-self.l2*np.sin(q[0]+q[1])
        j21=self.l1*np.cos(q[0])+self.l2*np.cos(q[0]+q[1])
        j22=self.l2*np.cos(q[0]+q[1])
        return np.array([[j11,j12],[j21,j22]])
    def ik(self, target, q0=None, iters=50):
        q = self.q.copy() if q0 is None else np.array(q0,dtype=np.float64)
        for _ in range(iters):
            e = target - self.fk(q)
            if np.linalg.norm(e)<1e-4: break
            J=self.jac(q)
            dq = np.linalg.pinv(J) @ e
            q = q + 0.5*dq
        return q
    def step(self, tau):
        # simplified double integrator with damping (no full manipulator inertia for clarity)
        ddq = tau - self.damping*self.dq
        self.dq = self.dq + self.dt*ddq
        self.q = self.q + self.dt*self.dq
        return self.q.copy(), self.fk()


In [ ]:

arm=TwoLinkArm()
# targets on a circle
ts=np.linspace(0,2*np.pi,200)
targets=np.stack([1.2*np.cos(ts), 1.2*np.sin(ts)], axis=1)

def simulate_pd(kp=40.0, kd=8.0):
    arm.reset([0.3,-0.2])
    xs=[]; errs=[]; torques=[]
    for i,tgt in enumerate(targets):
        q_des = arm.ik(tgt, q0=arm.q)
        e = q_des - arm.q
        de = 0.0 - arm.dq  # desire zero velocity mid waypoints
        tau = kp*e + kd*de
        arm.step(tau)
        xs.append(arm.fk()); errs.append(np.linalg.norm(arm.fk()-tgt)); torques.append(np.linalg.norm(tau))
    return np.array(xs), np.array(errs), np.array(torques)

def simulate_open_loop():
    # apply IK once at start only — fails to track
    arm.reset([0.3,-0.2])
    q_des=arm.ik(targets[0])
    xs=[]; errs=[]
    for tgt in targets:
        # constant torque 0 after setting initial q magically
        arm.q = q_des  # freeze — silly open loop "plan without feedback"
        xs.append(arm.fk()); errs.append(np.linalg.norm(arm.fk()-tgt))
        # no update of q_des
    return np.array(xs), np.array(errs)

t0=time.time()
xs_pd, err_pd, tau_pd = simulate_pd()
xs_bad, err_bad = simulate_open_loop()
# weaker PD
xs_w, err_w, _ = simulate_pd(kp=5.0, kd=1.0)
elapsed=time.time()-t0
print("PD mean err", err_pd.mean(), "weak PD", err_w.mean(), "openloop", err_bad.mean())


In [ ]:

fig, axes = plt.subplots(1,2,figsize=(10,4))
axes[0].plot(targets[:,0], targets[:,1], "k--", label="desired path", alpha=0.5)
axes[0].plot(xs_pd[:,0], xs_pd[:,1], label="PD track")
axes[0].plot(xs_w[:,0], xs_w[:,1], label="weak PD", alpha=0.8)
axes[0].set_aspect("equal"); axes[0].legend(); axes[0].set_title("End-effector path")
axes[0].set_xlabel("x"); axes[0].set_ylabel("y")
axes[1].plot(err_pd, label="PD")
axes[1].plot(err_w, label="weak PD")
axes[1].plot(err_bad, label="no feedback")
axes[1].set_xlabel("t"); axes[1].set_ylabel("||ee - target||"); axes[1].legend()
axes[1].set_title("Tracking error")
fig.tight_layout(); fig.savefig(OUT/"stage06_arm_tracking.png", dpi=120); plt.close(fig)

# also draw arm skeleton at a few frames
fig, ax = plt.subplots(figsize=(4,4))
for idx in [0,50,100,150,199]:
    q=TwoLinkArm().ik(targets[idx])
    p0=np.array([0.,0.])
    p1=np.array([np.cos(q[0]), np.sin(q[0])])
    p2=p1+np.array([np.cos(q[0]+q[1]), np.sin(q[0]+q[1])])
    ax.plot([p0[0],p1[0],p2[0]],[p0[1],p1[1],p2[1]], "-o", alpha=0.6)
ax.plot(targets[:,0], targets[:,1], "k--", alpha=0.3)
ax.set_aspect("equal"); ax.set_title("IK postures along path")
fig.tight_layout(); fig.savefig(OUT/"stage06_ik_postures.png", dpi=120); plt.close(fig)

payload={
  "ok": True,
  "stage":"06-classical-robotics",
  "title":"Grok-robotics-06-classical",
  "metrics":{
    "pd_mean_err": float(err_pd.mean()),
    "weak_pd_mean_err": float(err_w.mean()),
    "openloop_mean_err": float(err_bad.mean()),
  },
  "gpu": gpu_info,
  "elapsed_sec": elapsed,
  "concept": "FK/IK + PD feedback tracks trajectories without learning",
  "new_capability": "classical robot control baseline that RL must eventually beat or match in sample efficiency",
  "compare_to_previous": "Stages01-05 were decision-theoretic; Stage06 introduces embodiment geometry and feedback control",
}
assert payload["metrics"]["pd_mean_err"] < payload["metrics"]["weak_pd_mean_err"]
(OUT/"results_stage06.json").write_text(json.dumps(payload, indent=2))
print(json.dumps(payload, indent=2))
print("STAGE06_OK")
